In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, cross_val_score 
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import randint, uniform

In [6]:
df = pd.read_csv('../data/cleaned/cleaned_data.csv')

# prepare the data
df_train = df.dropna(subset=['Price'])
df_missing = df[df['Price'].isna()]

features = ['Rooms', 'Distance', 'Bathroom', 'Car', 'Landsize', 'Postcode', 'SaleYear','SaleMonth','HouseAge','LandsizePerRoom','PricePerSqm','RoomsPerBathroom','BuildingDensity']
X = df_train[features]
y = df_train['Price']

In [7]:
rf = RandomForestRegressor(random_state=42, n_jobs=2)  # Reduced from -1 to 2

# Reduced parameter search space to prevent memory issues
param_distributions = {
    'n_estimators': randint(50, 200),  # Reduced range
    'max_features': ['sqrt', 'log2', 0.6],  # Removed float values
    'max_depth': randint(10, 20),  # Reduced range
    'min_samples_split': randint(5, 15),  # Reduced range  
    'min_samples_leaf': randint(2, 8),  # Reduced range
    'bootstrap': [True, False]  # Only keep True to reduce combinations
}

# Use fewer iterations and jobs to reduce memory usage
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,  # Reduced from 50 to 20
    scoring=make_scorer(mean_absolute_error, greater_is_better=False),  # Fixed scoring
    cv=3,  # Reduced from 5 to 3 folds
    verbose=1,
    random_state=42,
    n_jobs=2  # Reduced from -1 to 2
)

print("Starting hyperparameter search with reduced parameters...")
random_search.fit(X, y)
print("Hyperparameter search completed!")

Starting hyperparameter search with reduced parameters...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Hyperparameter search completed!
Hyperparameter search completed!


In [8]:
print("\n--- RandomizedSearchCV Results ---")
print(f"Best parameters found: {random_search.best_params_}")
print(f"Best cross-validation MAE (negative): {random_search.best_score_:.2f}")
print(f"Best cross-validation MAE (positive): {-random_search.best_score_:.2f}")

# You can also get the best estimator and evaluate it further
best_rf_model = random_search.best_estimator_

print("Type of best_model:", type(best_rf_model))
print("with the best parameters found:", random_search.best_params_)

# Evaluate the best model's R2 and RMSE using cross_val_score for comparison
print("\n--- Best Model's Cross-Validated Performance ---")
best_r2_scores = cross_val_score(best_rf_model, X, y, cv=5, scoring='r2', n_jobs=-1)
best_rmse_scores = np.sqrt(-cross_val_score(best_rf_model, X, y, cv=5, scoring='neg_mean_squared_error', n_jobs=-1))

print(f"Mean CV R2 (Best RandomForest): {best_r2_scores.mean():.4f}")
print(f"Mean CV MAE (Best RandomForest): {-random_search.best_score_:.2f}") # Already have this from random_search.best_score_
print(f"Mean CV RMSE (Best RandomForest): {best_rmse_scores.mean():.2f}")

# --- Comparison to your initial RandomForest CV scores ---
print("\n--- Comparison to initial RandomForest CV scores ---")
print("Initial RandomForest Mean CV R2: 0.7221")
print("Initial RandomForest Mean CV MAE: 196055.96")
print("Initial RandomForest Mean CV RMSE: 335772.26")


--- RandomizedSearchCV Results ---
Best parameters found: {'bootstrap': False, 'max_depth': 14, 'max_features': 0.6, 'min_samples_leaf': 5, 'min_samples_split': 12, 'n_estimators': 84}
Best cross-validation MAE (negative): -30324.93
Best cross-validation MAE (positive): 30324.93
Type of best_model: <class 'sklearn.ensemble._forest.RandomForestRegressor'>
with the best parameters found: {'bootstrap': False, 'max_depth': 14, 'max_features': 0.6, 'min_samples_leaf': 5, 'min_samples_split': 12, 'n_estimators': 84}

--- Best Model's Cross-Validated Performance ---
Mean CV R2 (Best RandomForest): 0.9780
Mean CV MAE (Best RandomForest): 30324.93
Mean CV RMSE (Best RandomForest): 92894.76

--- Comparison to initial RandomForest CV scores ---
Initial RandomForest Mean CV R2: 0.7221
Initial RandomForest Mean CV MAE: 196055.96
Initial RandomForest Mean CV RMSE: 335772.26
Mean CV R2 (Best RandomForest): 0.9780
Mean CV MAE (Best RandomForest): 30324.93
Mean CV RMSE (Best RandomForest): 92894.76

-